# OP3 (Open Problems perturbation dataset) — download and plate-layout inspection

**Why this dataset:** a second plate-based single-cell perturbation screen in PBMCs, widely used as a benchmark (NeurIPS 2023 / Kaggle; GEO GSE279945). Each cell has its **plate, well, row and column**, and each plate has **several DMSO control wells**. This allows testing, in an independent dataset, the two Parse-only findings:
1. **well-to-well noise** (replicate DMSO wells, as the PBS wells in Parse)
2. **position / reference effects** (are control wells placed in particular rows or columns, and do they differ from other wells?)

and repeating the **shared-well coordination** audit.

Source: `s3://openproblems-bio/public/neurips-2023-competition/sc_counts_reannotated_with_counts.h5ad` (public bucket), obs columns include `well`, `row`, `col`, `plate_name`, `donor_id`, `sm_name`, `cell_type`, `split`; raw counts in `layers["counts"]`.

In [1]:
import os, subprocess
import numpy as np, pandas as pd
OP3_DIR = "data/op3"
os.makedirs(OP3_DIR, exist_ok=True)
URL = "https://openproblems-bio.s3.amazonaws.com/public/neurips-2023-competition/sc_counts_reannotated_with_counts.h5ad"
OUT = os.path.join(OP3_DIR, "sc_counts_reannotated_with_counts.h5ad")

head = subprocess.run(["curl", "-sIL", URL], capture_output=True, text=True).stdout
size = [int(l.split(":")[1]) for l in head.splitlines() if l.lower().startswith("content-length")]
print("remote size:", f"{size[-1] / 1e9:.2f} GB" if size else "unknown")

remote size: 3.52 GB


## 1. Download (resumable)

In [2]:
remote = size[-1] if size else None
for attempt in range(20):
    have = os.path.getsize(OUT) if os.path.exists(OUT) else 0
    if remote and have >= remote: break
    subprocess.run(["curl", "-L", "-C", "-", "--retry", "10", "--retry-delay", "20", "-sS", "-o", OUT, URL])
print(f"{os.path.getsize(OUT) / 1e9:.2f} GB")

3.52 GB


## 2. Inspect design and plate layout

In [4]:
import anndata as ad, h5py
a = ad.read_h5ad(OUT, backed="r")
obs = a.obs.copy()
with h5py.File(OUT, "r") as f:
    print(a.shape, "| layers in file:", list(f["layers"].keys()) if "layers" in f else "none")
print("obs columns:", obs.columns.tolist())
for col in ["donor_id", "plate_name", "cell_type", "split", "control"]:
    if col in obs: print(f"\n{col}:", obs[col].value_counts().to_dict())

# control wells are flagged in obs["control"] / split == "control" (the DMSO name is spelled out)
is_ctrl = obs["control"].astype(str).str.lower().isin(["true", "1"]) if "control" in obs else obs["split"] == "control"
print("\ncompounds in control wells:", obs.loc[is_ctrl, "sm_name"].value_counts().to_dict())

obs["is_ctrl"] = is_ctrl
wells = (obs.groupby(["plate_name", "well"], observed=True)
         .agg(donor=("donor_id", "first"), row=("row", "first"), col=("col", "first"),
              compound=("sm_name", "first"), is_ctrl=("is_ctrl", "first"), n_cells=("sm_name", "size")).reset_index())
wells.to_csv(os.path.join(OP3_DIR, "well_layout.csv"), index=False)
print("\ncontrol wells per plate and compound:\n", wells[wells.is_ctrl].groupby(["plate_name", "compound"], observed=True).size().unstack(fill_value=0))
for plate in sorted(wells.plate_name.unique())[:2]:
    w = wells[(wells.plate_name == plate) & wells.is_ctrl]
    print(f"\n{plate} control positions:", sorted(zip(w.row.astype(str), w.col.astype(int), w.compound.astype(str))))
print("\ncells per control well x cell type (median):")
print(obs[obs.is_ctrl].groupby(["plate_name", "well", "cell_type"], observed=True).size().groupby("cell_type").median())
print("\ncells per treated well x cell type (median):")
print(obs[~obs.is_ctrl].groupby(["plate_name", "well", "cell_type"], observed=True).size().groupby("cell_type").median())

(298087, 21265) | layers in file: []
obs columns: ['dose_uM', 'timepoint_hr', 'well', 'row', 'col', 'plate_name', 'cell_id', 'cell_type', 'split', 'donor_id', 'sm_name', 'control', 'SMILES', 'sm_lincs_id', 'library_id', 'leiden_res1', 'group', 'cell_type_orig', 'plate_well_celltype_reannotated', 'cell_count_by_well_celltype', 'cell_count_by_plate_well']

donor_id: {'Donor 2': 103026, 'Donor 3': 98673, 'Donor 1': 96388}

plate_name: {'NeurIPS2023-013': 55424, 'NeurIPS2023-014': 49684, 'NeurIPS2023-015': 48989, 'NeurIPS2023-010': 48259, 'NeurIPS2023-011': 48129, 'NeurIPS2023-012': 47602}

cell_type: {'T cells': 185202, 'B cells': 46386, 'Myeloid cells': 46080, 'NK cells': 20419}

split: {'train': 157555, 'control': 78840, 'private_test': 37956, 'public_test': 23736}

control: {False: 219247, True: 78840}

compounds in control wells: {'Dimethyl Sulfoxide': 28452, 'Dabrafenib': 26333, 'Belinostat': 24055, '5-(9-Isopropyl-8-methyl-2-morpholino-9H-purin-6-yl)pyrimidin-2-amine': 0, 'ABT737': 